In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split , GridSearchCV , RandomizedSearchCV\
                                    ,StratifiedKFold , KFold

from sklearn.linear_model import LogisticRegression , LinearRegression , Ridge , Lasso

from sklearn.metrics import mean_absolute_error, mean_squared_error,\
                            root_mean_squared_error, r2_score,\
                            accuracy_score, classification_report,\
                            confusion_matrix

from sklearn.naive_bayes import GaussianNB


from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import sys
import importlib


import Handle_Script
importlib.reload(Handle_Script)

from Handle_Script import handle_script



In [ ]:
data = pd.read_csv(r'Fifa.csv')
df = data.copy()
hs = handle_script()

In [ ]:
df =  hs.handle_data(df)

In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(18, 15))
ax = ax.flatten()

i = 0

for col in data.select_dtypes('number').columns:
    # الأصل
    sns.histplot(data[col], kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    # log transform (مع معالجة القيم =0)
    sns.histplot(np.log1p(data[col]), kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

    sns.boxplot(data[col], ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    # log transform (مع معالجة القيم =0)
    sns.boxplot(np.log1p(data[col]), ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

# نحذف أي subplots فاضية
for j in range(i, len(ax)):
    fig.delaxes(ax[j])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(18, 15))
ax = ax.flatten()

i = 0

for col in df.select_dtypes('number').columns:
    
    sns.histplot(df[col], kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1

    sns.histplot(np.log1p(df[col]), kde=True, ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1

    sns.boxplot(df[col], ax=ax[i])
    ax[i].set_title(f'{col} (Original)')
    i += 1


    sns.boxplot(np.log1p(df[col]), ax=ax[i])
    ax[i].set_title(f'{col} (Log)')
    i += 1


for j in range(i, len(ax)):
    fig.delaxes(ax[j])

plt.tight_layout()
plt.show()

In [ ]:
train , test = train_test_split(df , shuffle=True , random_state=42 , test_size=.2)
hs.fit_boxcox(train)
train = hs.transform_boxcox(train)
test = hs.transform_boxcox(test)

hs.fit_scaler(train)
train = hs.transform_scaler(train)
test = hs.transform_scaler(test)

hs.fit_encode(df)
train = hs.transform_encode(train)
test = hs.transform_encode(test)

In [ ]:
fig,ax = plt.subplots(1,2 , figsize = (10,6))
sns.histplot(test['Value Per M$_after_boxcox'], ax=ax[0])
ax[0].set_title('test')
sns.histplot(train['Value Per M$_after_boxcox'] , ax=ax[1])
ax[1].set_title('train')

In [ ]:
train = hs.selcet_features(train)
test = hs.selcet_features(test)

In [ ]:
plt.figure(figsize=(7,4))
sns.heatmap(train.select_dtypes('number').drop(columns=['Country_label', 'Position_label', 'Team_label']).corr() , annot=True , fmt='.1f' , cmap='Blues')

# Task 3 : Create Classification Target (Logistic Regression + Naïve Bayes)

* Use the dataset to classify players into 4 performance tiers based on Overall_Rating or Total_Stats Score

In [ ]:
train['performance_tiers'], bins = pd.qcut(
    train['Overall_Rating_scaled'],
    q=4,
    labels=['Low', 'Average', 'Good', 'Elite'],
    retbins=True,
    duplicates='drop'
)


In [ ]:
bins

In [ ]:
test['performance_tiers'] = pd.cut(
    test['Overall_Rating_scaled'],
    bins=bins,
    labels=['Low', 'Average', 'Good', 'Elite'],
    include_lowest=True
)

test['performance_tiers'] = test['performance_tiers'].fillna('Average')

* Show how many players fall into each category after splitting

In [ ]:
class_counts_train = train['performance_tiers'].value_counts().sort_values(ascending = False)
class_counts_test = test['performance_tiers'].value_counts().sort_values(ascending = False)

print('train : \n',class_counts_train , '\n')
print('test : \n' ,class_counts_test )

* Plot a bar chart showing the class distribution. Is it balanced or imbalanced?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 7))

class_counts_train.plot(kind='bar', ax=ax[0])
ax[0].set_title('Train')
ax[0].set_xlabel('Tier')
ax[0].set_ylabel('Number of Players')
ax[0].tick_params(axis='x', rotation=0)

class_counts_test.plot(kind='bar', ax=ax[1])
ax[1].set_title('Test')
ax[1].set_xlabel('Tier')
ax[1].set_ylabel('Number of Players')
ax[1].tick_params(axis='x', rotation=0)

fig.suptitle('Distribution of Performance Tiers')

plt.tight_layout()
plt.show()

* encode for Performance Tier

In [ ]:
map_Performance_Tier = {'Low' : 0, 'Average' : 1,
                        'Good' : 2, 'Elite' : 3}
train['Performance_Tier_encoded'] = (train['performance_tiers']).map(map_Performance_Tier)
test['Performance_Tier_encoded'] = (test['performance_tiers']).map(map_Performance_Tier)

# Task 4 : Model 1: Polynomial Regression

* Train regression models to predict the player market value

* Train a baseline Linear Regression model and evaluate it using: `MAE`, `MSE`, `RMSE`, and `R²` score on both train and test sets.

In [ ]:
y_train = train['Value Per M$_after_boxcox_scaled']
x_train = train.drop(columns=['performance_tiers', 'Performance_Tier_encoded', 'Value Per M$_after_boxcox_scaled'])

y_test = test['Value Per M$_after_boxcox_scaled']
x_test = test.drop(columns=['performance_tiers', 'Performance_Tier_encoded', 'Value Per M$_after_boxcox_scaled'])


In [ ]:
linear = LinearRegression()
linear.fit(x_train, y_train)

y_pred_train = linear.predict(x_train)
y_pred_test = linear.predict(x_test)

print("Train R2:", r2_score(y_train, y_pred_train))
print("Train MAE:", mean_absolute_error(y_train, y_pred_train))
print("Train MSE:", mean_squared_error(y_train, y_pred_train))
print("Train RMSE:", root_mean_squared_error(y_train, y_pred_train) , '\n')

print("Test R2:", r2_score(y_test, y_pred_test))
print("Test MAE:", mean_absolute_error(y_test, y_pred_test))
print("Test MSE:", mean_squared_error(y_test, y_pred_test))
print("Test RMSE:", root_mean_squared_error(y_test, y_pred_test))

* Use PolynomialFeatures to expand the numerical features only for degree = 1, 2, 3, and 4 if needed.

* For each degree, record Train R² and Test R². Plot degree vs. performance. What trend do you observe?

In [ ]:
num_cols = ['Age_scaled', 'Overall_Rating_scaled', 'Future Potential_scaled','Total_Stats Score_scaled', 'Value Per M$_after_boxcox_scaled']
cat_cols = ['Country_label', 'Position_label', 'Team_label', 'performance_tiers', 'Performance_Tier_encoded']
num_cols_train = ['Age_scaled', 'Overall_Rating_scaled', 'Future Potential_scaled','Total_Stats Score_scaled']

In [ ]:
degrees = [1, 2, 3, 4]

poly_num_cols = ColumnTransformer([
    ('poly_ct', PolynomialFeatures(degree=2, include_bias=False), num_cols_train)
    ],remainder = 'passthrough')

model_pipeline = Pipeline(steps=[
    ('poly_pl' , poly_num_cols),
    ('linear' , LinearRegression())

])

param_grid = {
    'poly_pl__poly_ct__degree' : degrees
}

kfold = KFold(n_splits=5 , shuffle=True , random_state=42)

model_grid = GridSearchCV(model_pipeline , param_grid=param_grid , scoring='r2' , cv=kfold , return_train_score=True)

model_grid.fit(x_train, y_train)


In [ ]:
results = model_grid.cv_results_
results

In [ ]:
results = model_grid.cv_results_

train_r2 = []
val_r2 = []

for i in range(len(results['params'])):
    print(results['params'][i])
    print("Train:", results['mean_train_score'][i])
    train_r2.append(results['mean_train_score'][i])

    print("Val:", results['mean_test_score'][i])
    val_r2.append(results['mean_test_score'][i])
    print("------")

In [ ]:
best_index_train = np.argmax(train_r2)
best_index_val = np.argmax(val_r2)

best_index_train , best_index_train


* for check if exist overfitting

In [ ]:
best_r2_train = train_r2[best_index_train]
best_r2_val = val_r2[best_index_val]

best_r2_train , best_r2_val

In [ ]:
best_degree_train = degrees[best_index_train]
best_degree_val = degrees[best_index_val]

best_degree_train , best_degree_val

In [ ]:
model_grid.best_estimator_

In [ ]:
model_grid.best_params_

* grid.best_score_ ==> best_score what come of `validation score`

In [ ]:
model_grid.best_score_

In [ ]:
print("Best Degree:", best_degree_val)
print("Best R2:", best_r2_val)

In [ ]:
plt.plot([1,2,3,4] , train_r2 , label = 'Train_r2')
plt.plot([1,2,3,4] , val_r2 , label = 'VAl_r2')

plt.xlabel('Polynomail Degree')
plt.ylabel('R2 Score')
plt.title('Polynomail Degree vs Performance')
plt.legend()
plt.show()

In [ ]:
degrees = [1, 2, 3, 4]

train_r2 = []
test_r2 = []

for d in degrees:

    poly_num_cols = ColumnTransformer(transformers=[
        ('poly_ct' ,  PolynomialFeatures(degree=d, include_bias=False) , num_cols_train)
        ],remainder='passthrough'
    )

    model = Pipeline([
        ('poly', poly_num_cols),
        ('lr', LinearRegression())
    ])
    
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    train_r2.append(r2_score(y_train, y_train_pred))
    test_r2.append(r2_score(y_test, y_test_pred))

    print(f"\nDegree {d}")
    print("Train R2:", r2_score(y_train, y_train_pred))
    print("Test R2:", r2_score(y_test, y_test_pred))

In [ ]:
best_index_train = np.argmax(train_r2)
best_index_test = np.argmax(test_r2)

best_index_train , best_index_test

* to check about overfitting
* we depend on best_r2_test

In [ ]:
best_r2_train = train_r2[best_index_train]
best_r2_test = train_r2[best_index_test]

best_r2_train , best_r2_test

In [ ]:
best_degree = degrees[best_index_test]
best_degree

In [ ]:
print("Best Degree:", best_degree)
print("Best R2:", best_r2_test)

In [ ]:
plt.plot(degrees, train_r2, label='Train R2')
plt.plot(degrees, test_r2, label='Test R2')

plt.xlabel("Polynomial Degree")
plt.ylabel("R2 Score")
plt.title("Polynomial Degree vs Performance")
plt.legend()
plt.show()

In [ ]:
degrees = [1, 2, 3, 4]

splits_train_r2 = {}
splits_val_r2 = {}

kfold = KFold(n_splits=5 , shuffle=True , random_state=42)

i=0

for train_idx , val_idx in kfold.split(train):
    
    xtr , xval = x_train.iloc[train_idx] , x_train.iloc[val_idx]
    ytr , yval = y_train.iloc[train_idx] , y_train.iloc[val_idx]

    fold_train_r2 = []
    fold_val_r2 = []

    print(f"\nFold {i}")
    
    for d in degrees:

        poly_num_cols = ColumnTransformer(transformers=[
            ('poly_ct' ,  PolynomialFeatures(degree=d, include_bias=False) , num_cols_train)
            ],remainder='passthrough'
        )

        model = Pipeline([
            ('poly', poly_num_cols),
            ('lr', LinearRegression())
        ])
        
        model.fit(xtr, ytr)

        y_train_pred = model.predict(xtr)
        y_val_pred = model.predict(xval)

        fold_train_r2.append(r2_score(ytr, y_train_pred))
        fold_val_r2.append(r2_score(yval, y_val_pred))

        
        print(f"\nDegree {d}")
        print("Train R2:", r2_score(ytr, y_train_pred))
        print("Val R2:", r2_score(yval, y_val_pred))


    splits_train_r2['split_'+str(i)] = fold_train_r2
    splits_val_r2['split_'+str(i)] = fold_val_r2
    
    i += 1



def mean_for_each_degree(splits_r2):
    idx = 0

    mean_for_each_degree_to_all_splits = []

    for i in range(len(splits_r2['split_0'])): # ==> 4 degree

        r2 = 0

        for split in splits_r2.values():
            r2 += split[idx]

        mean_for_each_degree_to_all_splits.append(r2/len(splits_r2))
        
        idx += 1

    return mean_for_each_degree_to_all_splits


mean_for_each_degree_to_all_folds_train_r2 = mean_for_each_degree(splits_train_r2)
mean_for_each_degree_to_all_folds_val_r2 = mean_for_each_degree(splits_val_r2)
    

In [ ]:
mean_for_each_degree_to_all_folds_train_r2

In [ ]:
mean_for_each_degree_to_all_folds_val_r2

* to check about overfitting

In [ ]:
best_index_train = np.argmax(mean_for_each_degree_to_all_folds_train_r2)
best_index_val = np.argmax(mean_for_each_degree_to_all_folds_val_r2)

best_index_train , best_index_val


In [ ]:
best_r2_train = mean_for_each_degree_to_all_folds_train_r2[best_index_train]
best_r2_val = mean_for_each_degree_to_all_folds_val_r2[best_index_val]

best_r2_train , best_r2_val

In [ ]:
best_degree_train = degrees[best_index_train]
best_degree_val = degrees[best_index_val]

best_degree_train , best_degree_val

* we depend on validation

In [ ]:
print("Best Degree:", best_degree_val)
print("Best R2:", best_r2_val)

In [ ]:
plt.plot(degrees, mean_for_each_degree_to_all_folds_train_r2, label='Train R2')
plt.plot(degrees, mean_for_each_degree_to_all_folds_val_r2, label='Test R2')

plt.xlabel("Polynomial Degree")
plt.ylabel("R2 Score")
plt.title("Polynomial Degree vs Performance")
plt.legend()
plt.show()

* Which degree gives the best generalization? Justify using the train/test gap.

* degree = `2`

* Regularization (Ridge and Lasso)
* Apply regularization on the best polynomial degree to control overfitting.

* Ridge (L2): Sweep alpha over a log-spaced range. Report Train and Test RMSE for each.

In [ ]:
alphas = np.logspace(-4, 4, 20)

In [ ]:
poly_num_cols = ColumnTransformer(transformers=[
    ('poly_ct' , PolynomialFeatures(degree=2, include_bias=False) , num_cols_train)
        ],remainder='passthrough'
    )

model_pipeline = Pipeline([
        ('poly_pl', poly_num_cols), 
        ('ridge', Ridge(alpha=.01))
    ])

param_grid = {'ridge__alpha' : alphas}

kfold = KFold(n_splits=5 , shuffle=True , random_state=42)

model_grid = GridSearchCV(model_pipeline , param_grid=param_grid , scoring='neg_root_mean_squared_error' , cv = kfold , return_train_score=True)

model_grid.fit(x_train , y_train)


In [ ]:
results = model_grid.cv_results_
results

In [ ]:
ridge_train_rmse = []
ridge_val_rmse = []

for idx , i in enumerate(range(len(results['params']))):
    print(results['params'][i] ,' : ',idx)
    print("Train:", results['mean_train_score'][i])
    ridge_train_rmse.append(results['mean_train_score'][i])

    print("Val:", results['mean_test_score'][i])
    ridge_val_rmse.append(results['mean_test_score'][i])
    print("------")

In [ ]:
best_index_train = np.argmax(ridge_train_rmse)
best_index_val = np.argmax(ridge_val_rmse)

best_index_train , best_index_val

* to chick about overfitting
* we depend on `rmse_val`

In [ ]:
best_rmse_train = ridge_train_rmse[best_index_train]
best_rmse_val = ridge_val_rmse[best_index_val]

best_rmse_train , best_rmse_val

* we depend on `best_alpha_val` this what give us best `rmse_val`

In [ ]:
best_alpha_train = alphas[best_index_train]
best_alpha_val = alphas[best_index_val]

best_alpha_train , best_alpha_val

In [ ]:
model_grid.best_estimator_

In [ ]:
model_grid.best_params_

In [ ]:
model_grid.best_score_

* we depend on `Validation`

In [ ]:
print("Best Alpha:", best_alpha_val)
print("Best Negative RMSE:", best_rmse_val)

In [ ]:
plt.plot(alphas , ridge_train_rmse , label = 'Train_RMSE')
plt.plot(alphas , ridge_val_rmse , label = 'VAl_RMSE')

plt.title("Ridge Regularization")
plt.ylabel('Negative Root Mean Square Error Score')
plt.xlabel('Alpha')
plt.xscale('log')  
plt.legend()
plt.show()

* As alpha increases, the model initially improves due to reduced overfitting. However, beyond a certain point, performance degrades due to underfitting. The best performance was achieved at a moderate alpha value (~4.28), where the validation RMSE was minimized. Ridge regularization helped stabilize the model without introducing a large train-test gap.

In [ ]:
ridge_train_rmse = []
ridge_test_rmse = []

for idx , a in enumerate(alphas):

    poly_num_cols = ColumnTransformer(transformers=[
        ('poly_ct' , PolynomialFeatures(degree=2, include_bias=False) , num_cols_train)
        ],remainder='passthrough'
    )

    model = Pipeline([
        ('poly_pl', poly_num_cols), 
        ('ridge', Ridge(alpha=a))
    ])

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    ridge_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    ridge_test_rmse.append(root_mean_squared_error(y_test, y_test_pred))

    
    print(f"Alpha {idx}:", a)
    print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
    print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred),'\n')

In [ ]:
best_index_rmse_train = np.argmin(ridge_train_rmse)
best_index_rmse_test = np.argmin(ridge_test_rmse)

best_index_rmse_train , best_index_rmse_test

* to check about overfitting
* depend on `rmse_test`

In [ ]:
best_rmse_train = ridge_train_rmse[best_index_rmse_train]
best_rmse_test = ridge_test_rmse[best_index_rmse_test]

best_rmse_train , best_rmse_test

In [ ]:
best_alpha_train = alphas[best_index_rmse_train]
best_alpha_test = alphas[best_index_rmse_test]

best_alpha_train , best_alpha_test

* We depend on test

In [ ]:
print("Best alpha:", best_alpha_test)
print("Best RMSE:", best_rmse_test)

In [ ]:
plt.plot(alphas , ridge_train_rmse , label = 'Train_RMSE')
plt.plot(alphas , ridge_test_rmse , label = 'Test_RMSE')

plt.xscale('log')

plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Ridge Regularization")
plt.legend()
plt.show()

In [ ]:
ridge_splits_train_rmse = {}
ridge_splits_val_rmse = {}

i=0

for train_idx , val_idx in kfold.split(train):
    xtr , xval = x_train.iloc[train_idx] , x_train.iloc[val_idx]
    ytr , yval = y_train.iloc[train_idx] , y_train.iloc[val_idx]

    ridge_fold_train_rmse = []
    ridge_fold_val_rmse = []

    for idx , a in enumerate(alphas):
        
        poly_num_cols = ColumnTransformer(transformers=[
            ('poly_ct' , PolynomialFeatures(degree=2, include_bias=False) , num_cols_train)
        ])

        model = Pipeline([
            ('poly', poly_num_cols), 
            ('ridge', Ridge(alpha=a))
        ])

        model.fit(xtr, ytr)

        y_train_pred = model.predict(xtr)
        y_val_pred = model.predict(xval)

        ridge_fold_train_rmse.append(root_mean_squared_error(ytr, y_train_pred))
        ridge_fold_val_rmse.append(root_mean_squared_error(yval, y_val_pred))

        
        print(f"Fold: {i}")
        print(f"Alpha {idx}:", a)
        print("Train RMSE:", root_mean_squared_error(ytr, y_train_pred))
        print("Test RMSE:", root_mean_squared_error(yval, y_val_pred),'\n')
    
    ridge_splits_train_rmse['split_'+str(i)] = ridge_fold_train_rmse
    ridge_splits_val_rmse['split_'+str(i)] = ridge_fold_val_rmse
    i += 1


def mean_for_each_alpha(splits_rmse):
    
    mean_for_each_alpha_to_all_splits = []

    for i in range( len(splits_rmse['split_0']) ): # ==> num. of alpha `20`

        rmse = 0

        for split in splits_rmse.values(): # ==> for each alpha have 5 value `5`
            rmse += split[i]
        
        mean_for_each_alpha_to_all_splits.append(rmse/len(splits_rmse))
    
    return mean_for_each_alpha_to_all_splits
    
mean_for_each_alpha_to_all_folds_train_rmse = mean_for_each_degree(ridge_splits_train_rmse)
mean_for_each_alpha_to_all_folds_val_rmse = mean_for_each_degree(ridge_splits_val_rmse)



In [ ]:
best_index_train = np.argmin(mean_for_each_alpha_to_all_folds_train_rmse)
best_index_val = np.argmin(mean_for_each_alpha_to_all_folds_val_rmse)

best_index_train , best_index_val


In [ ]:
best_rmse_train= mean_for_each_alpha_to_all_folds_train_rmse[best_index_train]
best_rmse_val= mean_for_each_alpha_to_all_folds_val_rmse[best_index_val]

best_rmse_train , best_rmse_val

In [ ]:
best_alpha_train = alphas[best_index_train]
best_alpha_val = alphas[best_index_val]

best_alpha_train , best_alpha_val

* we depend on Validation

In [ ]:
print("Best alpha:", best_alpha_val)
print("Best RMSE:", best_rmse_val)

In [ ]:
plt.plot(alphas ,mean_for_each_alpha_to_all_folds_train_rmse , label ='Train_RMSE' )
plt.plot(alphas ,mean_for_each_alpha_to_all_folds_val_rmse , label ='VAl_RMSE' )
plt.xscale('log')

plt.legend()
plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Ridge Regularization")

plt.tight_layout()
plt.show()

* Lasso (L1): Do the same.

In [ ]:
poly_num_cols = ColumnTransformer(transformers=[
    ('poly_ct' , PolynomialFeatures(degree=2, include_bias=False) , num_cols_train)
    ],remainder='passthrough'
)

model_pipeline = Pipeline([
        ('poly_pl', poly_num_cols), 
        ('lasso', Lasso(alpha=.1 , max_iter=1000))
    ])

param_grid = {'lasso__alpha' : alphas}


model_grid = GridSearchCV(model_pipeline , param_grid=param_grid , scoring='neg_root_mean_squared_error' , cv = kfold , return_train_score=True)

model_grid.fit(x_train , y_train)

In [ ]:
results = model_grid.cv_results_
results


In [ ]:
lasso_train_rmse = []
lasso_val_rmse = []

for idx , i in enumerate(range(len(results['params']))):
    print(results['params'][i],':',idx)
    print("Train:", results['mean_train_score'][i])
    lasso_train_rmse.append(results['mean_train_score'][i])

    print("Val:", results['mean_test_score'][i])
    lasso_val_rmse.append(results['mean_test_score'][i])
    print("------")


In [ ]:
best_index_train = np.argmax(lasso_train_rmse)
best_index_val = np.argmax(lasso_val_rmse)

best_index_train , best_index_val

In [ ]:
best_alpha_train = alphas[best_index_train]
best_alpha_val = alphas[best_index_val]

best_alpha_train , best_alpha_val

In [ ]:
best_rmse_train = lasso_train_rmse[best_index_train]
best_rmse_val =  lasso_val_rmse[best_index_val]

best_rmse_train , best_rmse_val

In [ ]:
model_grid.best_estimator_

In [ ]:
model_grid.best_params_

In [ ]:
model_grid.best_score_

In [ ]:
print("Best alpha:", best_alpha_val)
print("Best RMSE:", best_rmse_val)

In [ ]:
plt.plot(alphas , lasso_train_rmse , label = 'Train_RMSE')
plt.plot(alphas , lasso_val_rmse , label = 'VAl_RMSE')

plt.title("Lasso Regularization")
plt.ylabel('Negative Root Mean Square Error Score')
plt.xlabel('Alpha')
plt.xscale('log')  
plt.legend()
plt.show()

In [ ]:

lasso_train_rmse = []
lasso_test_rmse = []

for idx , a in enumerate(alphas):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('lasso', Lasso(alpha=a, max_iter=1000))
    ])

    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    lasso_train_rmse.append(root_mean_squared_error(y_train, y_train_pred))
    lasso_test_rmse.append(root_mean_squared_error(y_test, y_test_pred))

    print(f"\Alpha {idx} : {a}")
    print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
    print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred) , '\n')

In [ ]:
best_index_train = np.argmin(lasso_train_rmse)
best_index_test = np.argmin(lasso_test_rmse)

best_index_train , best_index_test

In [ ]:
best_alpha_train = alphas[best_index_train]
best_alpha_test = alphas[best_index_test]

best_alpha_train , best_alpha_test

In [ ]:
best_rmse_train = lasso_train_rmse[best_index_train]
best_rmse_test = lasso_test_rmse[best_index_test]

best_rmse_train , best_rmse_test

In [ ]:
print("Best alpha:", best_alpha_test)
print("Best RMSE:", best_rmse_test)

In [ ]:
plt.plot(alphas, lasso_train_rmse, label='Train RMSE')
plt.plot(alphas, lasso_test_rmse, label='Test RMSE')
plt.xscale('log')

plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Lasso Regularization")
plt.legend()
plt.show()

In [ ]:
lasso_splits_train_rmse = {}
lasso_splits_val_rmse = {}

i=0

for train_idx , val_idx in kfold.split(train):
    xtr , xval = x_train.iloc[train_idx] , x_train.iloc[val_idx]
    ytr , yval = y_train.iloc[train_idx] , y_train.iloc[val_idx]

    lasso_fold_train_rmse = []
    lasso_fold_val_rmse = []

    for idx , a in enumerate(alphas):
        
        poly_num_cols = ColumnTransformer(transformers=[
            ('poly_ct' , PolynomialFeatures(degree=2, include_bias=False) , num_cols_train)
        ])

        model = Pipeline([
            ('poly', poly_num_cols), 
            ('lasso', Lasso(alpha=a))
        ])

        model.fit(xtr, ytr)

        y_train_pred = model.predict(xtr)
        y_val_pred = model.predict(xval)

        lasso_fold_train_rmse.append(root_mean_squared_error(ytr, y_train_pred))
        lasso_fold_val_rmse.append(root_mean_squared_error(yval, y_val_pred))

        
        print(f"Fold: {i}")
        print(f"Alpha {idx}:", a)
        print("Train RMSE:", root_mean_squared_error(ytr, y_train_pred))
        print("Test RMSE:", root_mean_squared_error(yval, y_val_pred),'\n')
    
    lasso_splits_train_rmse['split_'+str(i)] = lasso_fold_train_rmse
    lasso_splits_val_rmse['split_'+str(i)] = lasso_fold_val_rmse
    i += 1


def mean_for_each_alpha(splits_rmse):
    
    mean_for_each_alpha_to_all_splits = []

    for i in range( len(splits_rmse['split_0']) ): # ==> num. of alpha `20`

        rmse = 0

        for split in splits_rmse.values(): # ==> for each alpha have 5 value `5`
            rmse += split[i]
        
        mean_for_each_alpha_to_all_splits.append(rmse/len(splits_rmse))
    
    return mean_for_each_alpha_to_all_splits
    
mean_for_each_alpha_to_all_folds_train_rmse = mean_for_each_degree(lasso_splits_train_rmse)
mean_for_each_alpha_to_all_folds_val_rmse = mean_for_each_degree(lasso_splits_val_rmse)



In [ ]:
best_index_train = np.argmin(mean_for_each_alpha_to_all_folds_train_rmse)
best_index_val = np.argmin(mean_for_each_alpha_to_all_folds_val_rmse)

best_index_train , best_index_val

In [ ]:
best_rmse_train= mean_for_each_alpha_to_all_folds_train_rmse[best_index_train]
best_rmse_val= mean_for_each_alpha_to_all_folds_val_rmse[best_index_val]

best_rmse_train , best_rmse_val

In [ ]:
best_alpha_train = alphas[best_index_train]
best_alpha_val = alphas[best_index_val]

best_alpha_train , best_alpha_val

In [ ]:
print("Best alpha:", best_alpha_val)
print("Best RMSE:", best_rmse_val)

In [ ]:
plt.plot(alphas ,mean_for_each_alpha_to_all_folds_train_rmse , label ='Train_RMSE' )
plt.plot(alphas ,mean_for_each_alpha_to_all_folds_val_rmse , label ='VAl_RMSE' )
plt.xscale('log')

plt.legend()
plt.xlabel("Alpha")
plt.ylabel("RMSE")
plt.title("Lasso Regularization")

plt.tight_layout()
plt.show()

* Plot both curves (alpha on log x-axis vs. Test RMSE). Mark the best alpha for each.

In [ ]:
poly_num_cols = ColumnTransformer(transformers=[
    ('poly_ct' , PolynomialFeatures(degree=2 , include_bias=False) , num_cols_train)
] , remainder='passthrough')

model_pipeline_ridge = Pipeline(steps=[
    ('poly_pl' , poly_num_cols),
    ('ridge' , Ridge(alpha=.1 ))
])

param_grid_ridge = {'ridge__alpha' : alphas}

model_ridge = GridSearchCV(model_pipeline_ridge , param_grid=param_grid_ridge, scoring='neg_root_mean_squared_error',cv = kfold , return_train_score=True)

model_ridge.fit(x_train , y_train)


In [ ]:
rmse_val_for_each_alpha_ridge = model_ridge.cv_results_['mean_test_score']
rmse_val_for_each_alpha_ridge

In [ ]:
best_alpha_ridge = model_ridge.best_params_['ridge__alpha']
best_alpha_ridge

In [ ]:
best_rmse_ridge = model_ridge.best_score_
best_rmse_ridge

In [ ]:
model_pipeline_lasso = Pipeline(steps=[
    ('poly_pl' , poly_num_cols),
    ('lasso' , Lasso(alpha=.0001))
])

param_grid_lasso = {'lasso__alpha' : alphas}

model_lasso = GridSearchCV(model_pipeline_lasso , param_grid=param_grid_lasso, scoring='neg_root_mean_squared_error',cv = kfold , return_train_score=True)

model_lasso.fit(x_train , y_train)


In [ ]:
rmse_val_for_each_alpha_lasso= model_lasso.cv_results_['mean_test_score']
rmse_val_for_each_alpha_lasso

In [ ]:
best_alpha_lasso = model_lasso.best_params_['lasso__alpha']
best_alpha_lasso

In [ ]:
best_rmse_lasso = model_lasso.best_score_
best_rmse_lasso

In [ ]:
plt.plot(alphas, rmse_val_for_each_alpha_ridge ,label = 'Ridge RMSE' )
plt.plot(alphas, rmse_val_for_each_alpha_lasso , label = 'Lasso RMSE')

plt.scatter(best_alpha_ridge, best_rmse_ridge, s=70, marker='x', color='red', zorder=5  )
plt.scatter(best_alpha_lasso, best_rmse_lasso, s=70, marker='o', color='blue', zorder=5  )

plt.annotate(f'Ridge best\nα={best_alpha_ridge:.4f}',
             (best_alpha_ridge, best_rmse_ridge),
             textcoords="offset points",
             xytext=(-10,-30))

plt.annotate(f'Lasso best\nα={best_alpha_lasso:.4f}',
             (best_alpha_lasso, best_rmse_lasso),
             textcoords="offset points",
             xytext=(-10,-30))

plt.xscale('log')
plt.legend()
plt.xlabel('Alpha')
plt.ylabel("RMSE")
plt.title('Ridge vs Lasso')

plt.tight_layout()
plt.show()

* Ridge performs slightly better than Lasso on this dataset.
The best RMSE for Ridge is -0.19069, while for Lasso it is -0.19072.

* This indicates that Ridge provides better generalization.

* The reason is that Ridge keeps all features and only shrinks their coefficients, while Lasso tends to eliminate features by setting some coefficients to zero.

* In this dataset, most features appear to be useful, so removing them (as Lasso does) harms performance.

* Additionally, Lasso shows a sharp performance drop for larger alpha values, indicating that aggressive regularization removes important features.

* Both models perform similarly, but Ridge is slightly better.

 -------------------------------------------------

* Did Lasso zero out any features? If yes, which ones and what does that imply?

In [ ]:
coef = model_lasso.best_estimator_.named_steps['lasso'].coef_
coef

In [ ]:
zero_features = np.sum(coef == 0)
print("Zero coefficients:", zero_features)

In [ ]:
df.head()

# Task 5


In [ ]:
x_train = train.drop(['Overall_Rating_scaled', 'performance_tiers' , 'Performance_Tier_encoded'], axis=1) 
y_train = train['Performance_Tier_encoded']

x_test = test.drop(['Overall_Rating_scaled', 'performance_tiers' , 'Performance_Tier_encoded'], axis=1) 
y_test = test['Performance_Tier_encoded']

baseline_lr = LogisticRegression(max_iter=1000)
baseline_lr.fit(x_train, y_train)



In [ ]:
x_train

In [ ]:
y_pred = baseline_lr.predict(x_test)
print("Baseline Logistic Regression Report:")
print(classification_report(y_test, y_pred))
baseline_acc = accuracy_score(y_test , y_pred)
baseline_acc

In [ ]:

plt.figure(figsize=(10, 7))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Baseline Model')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
c_range = np.logspace(-3, 3, 7)
train_accuracies = []
test_accuracies = []


In [ ]:
for c in c_range:
    lr = LogisticRegression(C=c, max_iter=1000)
    lr.fit(x_train, y_train)
    train_accuracies.append(accuracy_score(y_train, lr.predict(x_train)))
    test_accuracies.append(accuracy_score(y_test, lr.predict(x_test)))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(c_range, train_accuracies, label='Train Accuracy', marker='o')
plt.plot(c_range, test_accuracies, label='Test Accuracy', marker='o')
plt.xscale('log')
plt.xlabel('C value (Inverse of regularization strength)')
plt.ylabel('Accuracy')
plt.title('Effect of C on Model Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

best_c = c_range[np.argmax(test_accuracies)]
print(f"Best C value based on Test Accuracy: {best_c}")


In [ ]:
l1_model = LogisticRegression(penalty='l1', C=best_c, solver='saga', max_iter=5000)
l1_model.fit(x_train, y_train)
l1_score = accuracy_score(y_test, l1_model.predict(x_test))

In [ ]:
l2_model = LogisticRegression(penalty='l2', C=best_c, solver='lbfgs', max_iter=5000)
l2_model.fit(x_train, y_train)
l2_score = accuracy_score(y_test, l2_model.predict(x_test))

print(f"Accuracy with L1 (Saga) at C={best_c}: {l1_score:.4f}")
print(f"Accuracy with L2 (Lbfgs) at C={best_c}: {l2_score:.4f}")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train, y_train)
knn_pred = knn_model.predict(x_test)

In [ ]:
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

In [ ]:
svm_model = SVC(kernel='rbf', C=0.1, probability=True)
svm_model.fit(x_train, y_train)
svm_pred = svm_model.predict(x_test)

In [ ]:
svm_model_Linear = LinearSVC(penalty='l2' , C= 0.1 , multi_class= 'ovr')
svm_model_Linear.fit(x_train, y_train)
svm_Linear_pred = svm_model_Linear.predict(x_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)
rf_pred = rf_model.predict(x_test)

In [ ]:
models = {
    "k-Nearest Neighbors": knn_pred,
    "Support Vector Machine": svm_pred,
    "Support Vector Machine 2" : svm_Linear_pred ,
    "Random Forest": rf_pred
    }

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
for name, pred in models.items():
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred))

In [ ]:
svm_model_poly = SVC(kernel='poly', degree=3 , coef0=1 , gamma ='scale' , C=0.1, probability=True)
svm_model_poly.fit(x_train, y_train)
svm_pred = svm_model_poly.predict(x_test)

In [ ]:
accuracy_score(y_test ,svm_pred)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier


In [ ]:
dt_params = {
    'max_depth': [3, 5, 10, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring='accuracy')
grid_dt.fit(x_train, y_train)

print(f"Best DT Params: {grid_dt.best_params_}")
best_dt = grid_dt.best_estimator_

In [ ]:
rf_optimized = RandomForestClassifier(
    n_estimators=200,          
    criterion='gini',       
    max_depth=10,               
    min_samples_split=2,       
    random_state=42,
    n_jobs=-1                   
)

rf_optimized.fit(x_train, y_train)

y_pred_rf = rf_optimized.predict(x_test)

print(f"Optimized RF Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nDetailed Report:\n", classification_report(y_test, y_pred_rf))



In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='YlGnBu')
plt.title('Optimized Random Forest Confusion Matrix')
plt.show()

In [ ]:
train_acc = rf_optimized.score(x_train, y_train)
test_acc = rf_optimized.score(x_test, y_test)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc:.4f}")

if train_acc > test_acc + 0.10:
    print("Diagnosis: High Variance (Overfitting) - Need to reduce max_depth or increase regularization")
elif train_acc < 0.70:
    print("Diagnosis: High Bias (Underfitting) - Need more complex features or deeper trees")
else:
    print("Diagnosis: Balanced Model - Good tradeoff between Bias and Variance")

In [ ]:
from sklearn.model_selection import learning_curve

In [ ]:
from sklearn.ensemble import VotingClassifier
committee_model = VotingClassifier(
    estimators=[
        ('dt', best_dt),
        ('knn', knn_model),
        ('svc', svm_model)
    ],
    voting='soft'
)

committee_model.fit(x_train, y_train)
y_pred_ensemble = committee_model.predict(x_test)

print("Ensemble Model Accuracy:", accuracy_score(y_test, y_pred_ensemble))
print(classification_report(y_test, y_pred_ensemble))

In [ ]:
def plot_learning_curve(estimator, title, X, y):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5))
    
    train_scores_mean = np.mean(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)

    plt.plot(train_sizes, train_scores_mean, 'o-', label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-', label="Cross-validation score")
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    plt.legend(loc="best")
    plt.grid()

plt.figure(figsize=(10, 6))
plot_learning_curve(best_dt, "Learning Curve (Decision Tree)", x_train, y_train)
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
regressor = Ridge(alpha=1.0) 
classifier = rf_optimized   

class FIFAUnifiedSystem:
    def __init__(self, regression_model, classification_model):
        self.reg_model = regression_model
        self.clf_model = classification_model
        
    def predict_player(self, player_data):
        valuation = self.reg_model.predict(player_data)
        tier = self.clf_model.predict(player_data)
        
        return valuation, tier

fifa_system = FIFAUnifiedSystem(regressor.fit(x_train, y_train), rf_optimized)

In [ ]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(rf_optimized, x_train, y_train, cv=5)

print(f"Mean Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation (Stability): {cv_scores.std():.4f}")

if cv_scores.std() < 0.02:
    print("Statistical Proof: The system is HIGHLY STABLE across different data subsets")
else:
    print("Statistical Proof: The system shows some variance, consider more regularization")

In [ ]:
advanced_acc = accuracy_score(y_test, y_pred_rf)
comparison_data = {
    'Model': ['Baseline (Logistic)', 'Advanced (Optimized RF)'],
    'Accuracy': [baseline_acc, advanced_acc]
}

comp_df = pd.DataFrame(comparison_data)
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Accuracy', data=comp_df, palette='viridis')
plt.ylim(0.8, 1.0)
plt.title('Performance Jump: Baseline vs Advanced System')
for i, v in enumerate(comp_df['Accuracy']):
    plt.text(i, v + 0.005, f"{v:.4f}", ha='center')
plt.show()